# Part 01 개발 환경 구성 및 점검

교재 Part 01 4절의 실습 노트북입니다. STEP 1부터 차례로 실행하고, 각 STEP의 **확인 포인트**를 점검표(`templates/t01_env_check.md`)에 기록합니다.

**준비:** Hailo Developer Zone에서 내려받은 DFC 설치 파일을 본인 Google Drive의 `내 드라이브/hailo/` 폴더에 올려 두세요.

## STEP 1. Colab 실행 환경 확인 [코드 1-1]

In [ ]:
#- 모듈로딩
import sys, platform
#- 시스템 파일에서 전체 메모리(kB)를 읽어 GB로 환산
mem_gb = int(open("/proc/meminfo").read().split("MemTotal:")[1].split()[0]) / 1024**2

#- 기본 정보 출력
print(f"Python  : {sys.version.split()[0]}")
print(f"아키텍처: {platform.machine()}")
print(f"메모리  : {mem_gb:.1f} GB")

# 확인 포인트 -------------------------------------------------------------------------
# - 아키텍처 x86_64. Python은 3.12 이상(집필 기준 3.13)
# - DFC가 지원하는 3.10과 다르다는 점을 확인

## STEP 2. Drive 연결과 설치 파일 확인 [코드 1-2]

권한 요청 화면("Google Drive for desktop")이 나타나면 **모두 허용**합니다.

In [ ]:
#- 모듈로딩
import os
from google.colab import drive

#- Google Drive 연결 (권한 허용 창이 뜨면 승인)
drive.mount('/content/drive')

#- DFC 설치 파일명
DFC_FILE = 'hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl'

#- 내 드라이브의 DFC 설치 파일 경로 지정과 존재 확인
WHL = f"/content/drive/MyDrive/hailo/{DFC_FILE}"
print(f"파일 존재: {os.path.exists(WHL)}")

# 확인 포인트 -------------------------------------------------------------------------
# - 파일 존재 True 출력 체크
# - 버전이 다른 파일을 받았다면 DFC_FILE 값을 수정

## STEP 3. Python 3.10 전용 환경 생성과 DFC 설치 [코드 1-3]

약 75초 걸립니다. 세션이 끊기면 전용 환경도 사라지므로 이 셀부터 다시 실행합니다.

In [ ]:
#- 가상환경 도구 uv 설치
!pip install -q uv

#- DFC 전용 Python 3.10 가상환경 생성
!uv venv --clear --python 3.10 /content/dfc310

#- 전용 환경에 DFC 설치 (약 75초, 출력은 마지막 3줄만 표시)
!uv pip install --python /content/dfc310/bin/python "$WHL" 2>&1 | tail -n 3

# 확인 포인트 -------------------------------------------------------------------------
# - "Using CPython 3.10.x"와 설치된 패키지 목록 출력 체크
# - warning으로 시작하는 줄은 Colab 설정에 따른 안내로 무시 가능

## STEP 4. DFC 동작 확인 [코드 1-4]

In [ ]:
#- 전용 환경의 Python 버전 확인
!/content/dfc310/bin/python --version

#- 전용 환경에서 실행할 코드 (문자열: 들여쓰기 없이 작성)
COMMAND = '''
import hailo_sdk_client as h
print('DFC', h.__version__)
'''

#- 전용 환경(3.10)의 파이썬으로 실행: 노트북 기본 파이썬(3.13)에는 DFC가 없음
!/content/dfc310/bin/python -c "$COMMAND"

# 확인 포인트 -------------------------------------------------------------------------
# - Python 3.10.x와 DFC 3.34.0(받은 파일의 버전) 출력 체크
# - "No GPU chosen"(CPU로 동작), "Cannot use graphviz"(시각화 기능 생략)는 정상 안내

## STEP 5. 에뮬레이터 호출 점검 [코드 1-5]

In [ ]:
#- 전용 환경에서 실행할 코드 (문자열: 들여쓰기 없이 작성)
COMMAND = '''
from hailo_sdk_client import ClientRunner, InferenceContext as C
ClientRunner(hw_arch='hailo8')
print('EMULATOR_READY', [c.name for c in C])
'''

#- 전용 환경의 파이썬으로 실행해 에뮬레이터 준비 상태 점검
!/content/dfc310/bin/python -c "$COMMAND"

# 확인 포인트 -------------------------------------------------------------------------
# - EMULATOR_READY와 에뮬레이션 모드 이름(SDK_NATIVE, SDK_FP_OPTIMIZED 등) 출력 체크

## STEP 6. 실습 저장소 클론과 구조 확인 [코드 1-6]

In [ ]:
#- 실습 저장소 클론
!git clone https://github.com/npu-ondevice-ai/npu-course.git

#- 저장소 폴더로 이동해 구조 확인
%cd npu-course
!ls
# 확인 포인트 -------------------------------------------------------------------------
# - notebooks, data, models, templates 폴더 표시 체크

## 마무리

모든 STEP의 확인 포인트가 정상이라면 `templates/t01_env_check.md`에 결과를 기록해 제출합니다.